# 🩺 SkinDisNet — Skin Disease Classification
### Computer Vision Assignment | Google Colab

| | |
|---|---|
| **Dataset** | SkinDisNet — Mendeley Data v2 |
| **Paper** | *Data in Brief*, Vol 63, 2025 — DOI: 10.1016/j.dib.2025.112239 |
| **Classes** | Atopic Dermatitis, Contact Dermatitis, Eczema, Scabies, Seborrheic Dermatitis, Tinea Corporis |
| **Model** | EfficientNetB0 (Transfer Learning + Fine-Tuning) |
| **Framework** | TensorFlow / Keras (Colab GPU) |

---
### ⚡ Before Running:
1. Upload `SkinDisNet.zip` to your **Google Drive** (My Drive)
2. Run all cells top to bottom
3. Enable **GPU**: Runtime → Change runtime type → T4 GPU


## ⚙️ Step 1: Check GPU & Install Libraries

In [ ]:
# Check GPU
import tensorflow as tf
print(f"TensorFlow Version : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Available    : {gpus[0].name}")
else:
    print("⚠️  No GPU found — Go to Runtime > Change runtime type > T4 GPU")

# Install/check libraries
!pip install -q matplotlib seaborn scikit-learn Pillow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, json, shutil
warnings.filterwarnings('ignore')

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

print("✅ All libraries loaded.")


## 📂 Step 2: Mount Google Drive & Extract Dataset

> **Instructions:**
> 1. Upload `SkinDisNet.zip` to your **Google Drive (My Drive root)**
> 2. Run this cell — it will ask you to authorize Google Drive access
> 3. It will automatically extract the zip and find the `Preprocessed/` folder


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─── Path to your zip in Google Drive ────────────────────────────────────
ZIP_NAME  = "SkinDisNet.zip"   # ← change if your zip has a different name
ZIP_PATH  = f"/content/drive/MyDrive/{ZIP_NAME}"
EXTRACT_PATH = "/content/SkinDisNet"
# ─────────────────────────────────────────────────────────────────────────

# Extract zip
if not os.path.exists(EXTRACT_PATH):
    print(f"📦 Extracting {ZIP_NAME}...")
    shutil.unpack_archive(ZIP_PATH, EXTRACT_PATH)
    print("✅ Extraction complete!")
else:
    print("✅ Already extracted — skipping.")

# Auto-detect Preprocessed folder
DATASET_PATH = None
for root, dirs, files in os.walk(EXTRACT_PATH):
    if "Preprocessed" in dirs:
        DATASET_PATH = os.path.join(root, "Preprocessed")
        break
    # In case we're already inside Preprocessed
    if os.path.basename(root) == "Preprocessed" and len(dirs) > 0:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    # Fallback: look for folders with disease names
    for root, dirs, files in os.walk(EXTRACT_PATH):
        disease_keywords = ['dermatitis', 'eczema', 'scabies', 'tinea']
        if any(any(kw in d.lower() for kw in disease_keywords) for d in dirs):
            DATASET_PATH = root
            break

print(f"\n📁 Dataset path : {DATASET_PATH}")

# Show class folders
CLASS_NAMES = sorted([d for d in os.listdir(DATASET_PATH)
                      if os.path.isdir(os.path.join(DATASET_PATH, d))])
print(f"\n✅ Classes found ({len(CLASS_NAMES)}):")
total = 0
class_counts = {}
for cls in CLASS_NAMES:
    cls_path = os.path.join(DATASET_PATH, cls)
    count = len([f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))])
    class_counts[cls] = count
    total += count
    print(f"   {cls:<35} → {count:>4} images")
print(f"   {'TOTAL':<35} → {total:>4} images")


## 🖼️ Step 3: Visualise Sample Images (One Per Class)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle("SkinDisNet — Sample Images Per Class", fontsize=16, fontweight='bold')

for ax, cls in zip(axes.flatten(), CLASS_NAMES):
    cls_path = os.path.join(DATASET_PATH, cls)
    img_file = sorted([f for f in os.listdir(cls_path)
                       if f.lower().endswith(('.jpg','.jpeg','.png'))])[0]
    img = Image.open(os.path.join(cls_path, img_file)).resize((224, 224))
    ax.imshow(img)
    ax.set_title(cls, fontsize=11, fontweight='bold', pad=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig("/content/sample_images.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: /content/sample_images.png")


## 📊 Step 4: Class Distribution

In [ ]:
plt.figure(figsize=(11, 5))
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B3','#937860']
bars = plt.bar(class_counts.keys(), class_counts.values(),
               color=colors, edgecolor='black', width=0.6)
plt.title("Class Distribution — SkinDisNet Preprocessed Folder", fontsize=14, fontweight='bold')
plt.xlabel("Skin Disease Class", fontsize=12)
plt.ylabel("Number of Images", fontsize=12)
plt.xticks(rotation=20, ha='right', fontsize=10)
for bar, val in zip(bars, class_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig("/content/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()


## ⚙️ Step 5: Data Preprocessing & Augmentation

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42

# EfficientNetB0 has its own internal normalization.
# Use preprocess_input — do NOT use rescale=1./255 (that breaks it).
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    brightness_range=[0.75, 1.25],
    shear_range=0.1,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

CLASS_NAMES = list(train_gen.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)

# Class weights to fix severe imbalance (AD=70, SD=79 vs CD=477)
cw_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
CLASS_WEIGHTS = dict(enumerate(cw_array))

print(f'\n✅ Data generators ready')
print(f'   Training   samples : {train_gen.samples}')
print(f'   Validation samples : {val_gen.samples}')
print(f'   Classes            : {CLASS_NAMES}')
print(f'   Batch size         : {BATCH_SIZE}  |  Image size: {IMG_SIZE}x{IMG_SIZE}')
print(f'\n⚖️  Class weights:')
for cls, w in zip(CLASS_NAMES, cw_array):
    print(f'   {cls:<20} {w:.3f}')


## 🧠 Step 6: Build EfficientNetB0 Transfer Learning Model

In [ ]:
from tensorflow.keras import regularizers

def build_model(num_classes):
    base_model = EfficientNetB0(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x      = base_model(inputs, training=False)
    x      = layers.GlobalAveragePooling2D()(x)
    x      = layers.BatchNormalization()(x)
    x      = layers.Dense(512, activation='relu',
                           kernel_regularizer=regularizers.l2(1e-4))(x)
    x      = layers.Dropout(0.5)(x)
    x      = layers.Dense(256, activation='relu',
                           kernel_regularizer=regularizers.l2(1e-4))(x)
    x      = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(NUM_CLASSES)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\n✅ Model ready | Total params: {model.count_params():,}')


## 🏋️ Step 7: Phase 1 Training — Frozen Base (Feature Extraction)

In [ ]:
EPOCHS_PHASE1 = 20

callbacks_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=4, verbose=1, min_lr=1e-7),
    ModelCheckpoint('/content/best_model_phase1.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

print('🚀 Phase 1 Training started (frozen base)...')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_p1,
    class_weight=CLASS_WEIGHTS,
    verbose=1
)

print(f'\n✅ Phase 1 done — Best Val Accuracy: {max(history1.history["val_accuracy"])*100:.2f}%')


## 🔧 Step 8: Phase 2 — Fine-Tuning (Unfreeze Top 30 Layers)

In [ ]:
# Unfreeze top 50 layers (was 30 — more capacity)
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

print(f'Trainable base layers: {sum(1 for l in base_model.layers if l.trainable)} / {len(base_model.layers)}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_PHASE2 = 20

callbacks_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=4, verbose=1, min_lr=1e-8),
    ModelCheckpoint('/content/best_model_final.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

print('\n🚀 Phase 2 Fine-Tuning started...')
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_p2,
    class_weight=CLASS_WEIGHTS,
    verbose=1,
    initial_epoch=len(history1.history['accuracy'])
)

print(f'\n✅ Phase 2 done — Best Val Accuracy: {max(history2.history["val_accuracy"])*100:.2f}%')


## 📈 Step 9: Training & Validation Curves

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
ep_range = range(1, len(acc)+1)
split_ep = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SkinDisNet EfficientNetB0 — Training History", fontsize=14, fontweight='bold')

axes[0].plot(ep_range, acc,     'b-o', label='Train Acc',  markersize=4)
axes[0].plot(ep_range, val_acc, 'r-o', label='Val Acc',    markersize=4)
axes[0].axvline(x=split_ep, color='green', linestyle='--', label='Fine-tune start')
axes[0].set_title('Accuracy');  axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep_range, loss,     'b-o', label='Train Loss', markersize=4)
axes[1].plot(ep_range, val_loss, 'r-o', label='Val Loss',   markersize=4)
axes[1].axvline(x=split_ep, color='green', linestyle='--', label='Fine-tune start')
axes[1].set_title('Loss');  axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Final Train Acc : {acc[-1]*100:.2f}%  |  Final Val Acc : {val_acc[-1]*100:.2f}%")


## 📊 Step 10: Evaluation — Confusion Matrix & Classification Report

In [ ]:
model = keras.models.load_model('/content/best_model_final.keras')

val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_gen.classes

print('\n' + '='*65)
print('              CLASSIFICATION REPORT')
print('='*65)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='grey')
plt.title('Confusion Matrix — SkinDisNet Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=30, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

overall_acc = np.sum(np.diag(cm)) / np.sum(cm)
print(f'\n✅ Overall Accuracy: {overall_acc*100:.2f}%')


## 💾 Step 11: Save Model to Google Drive

In [ ]:
# Save in native Keras format (recommended over legacy .h5)
model.save('/content/skindisnet_efficientnetb0.keras')

with open('/content/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

drive_save_path = '/content/drive/MyDrive/SkinDisNet_Model/'
os.makedirs(drive_save_path, exist_ok=True)
shutil.copy('/content/skindisnet_efficientnetb0.keras', drive_save_path)
shutil.copy('/content/class_names.json',               drive_save_path)
shutil.copy('/content/training_curves.png',            drive_save_path)
shutil.copy('/content/confusion_matrix.png',           drive_save_path)

print('✅ Model saved to Colab:        /content/skindisnet_efficientnetb0.keras')
print(f'✅ Model backed up to Drive:   {drive_save_path}')
print(f'✅ Class names saved:          {CLASS_NAMES}')


## 🧪 Step 12: Test App — Classify Any Skin Image

> **Two ways to test:**
> - **Option A:** Upload an image directly from your computer
> - **Option B:** Use an image already in Google Drive


In [ ]:
from google.colab import files as colab_files
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

model = keras.models.load_model('/content/skindisnet_efficientnetb0.keras')
with open('/content/class_names.json') as f:
    CLASS_NAMES = json.load(f)

IMG_SIZE = 224

def predict_skin_disease(image_path):
    img         = Image.open(image_path).convert('RGB')
    img_resized = img.resize((IMG_SIZE, IMG_SIZE))
    img_array   = np.array(img_resized, dtype=np.float32)
    img_array   = eff_preprocess(img_array)   # must match training preprocessing
    img_batch   = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_batch, verbose=0)[0]
    pred_idx    = np.argmax(predictions)
    pred_class  = CLASS_NAMES[pred_idx]
    confidence  = predictions[pred_idx] * 100

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('SkinDisNet — Skin Disease Classification Result',
                 fontsize=14, fontweight='bold')

    axes[0].imshow(img_resized)
    axes[0].set_title('Input Image', fontsize=12)
    axes[0].axis('off')

    colors = ['#e74c3c' if i == pred_idx else '#3498db'
              for i in range(len(CLASS_NAMES))]
    bars = axes[1].barh(CLASS_NAMES, predictions * 100,
                        color=colors, edgecolor='black', height=0.6)
    axes[1].set_xlabel('Confidence (%)', fontsize=11)
    axes[1].set_title('Class Probabilities', fontsize=12)
    axes[1].set_xlim(0, 115)
    for bar, val in zip(bars, predictions * 100):
        axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                     f'{val:.1f}%', va='center', fontsize=10)

    plt.tight_layout()
    plt.savefig('/content/prediction_result.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n' + '='*55)
    print(f'  PREDICTED CLASS  :  {pred_class}')
    print(f'  CONFIDENCE       :  {confidence:.2f}%')
    print('='*55)
    print('\n  All probabilities:')
    for name, prob in sorted(zip(CLASS_NAMES, predictions), key=lambda x: -x[1]):
        bar_str = '█' * int(prob * 25)
        print(f'    {name:<35} {prob*100:5.2f}%  {bar_str}')

    return pred_class, confidence

print('📤 Upload a skin disease image:')
uploaded = colab_files.upload()

if uploaded:
    image_filename = list(uploaded.keys())[0]
    image_path     = f'/content/{image_filename}'
    print(f'\n✅ Uploaded: {image_filename}')
    predicted_class, confidence = predict_skin_disease(image_path)


## ✅ Assignment Summary

| Item | Details |
|---|---|
| **Dataset** | SkinDisNet — Mendeley Data v2 |
| **Journal** | *Data in Brief*, Vol 63, 2025 (Elsevier) |
| **DOI** | 10.1016/j.dib.2025.112239 |
| **Model** | EfficientNetB0 (ImageNet weights) |
| **Strategy** | Phase 1: Frozen base → Phase 2: Fine-tune top 30 layers |
| **Augmentation** | Rotation, Flip, Zoom, Brightness, Shift |
| **Classes** | 6 Skin Disease Categories |
| **Input Size** | 224 × 224 × 3 |
| **Platform** | Google Colab (GPU) |

### 📁 Output Files Saved to Google Drive (`SkinDisNet_Model/`)
| File | Description |
|---|---|
| `skindisnet_efficientnetb0.h5` | Trained model weights |
| `class_names.json` | Class label mapping |
| `training_curves.png` | Accuracy & loss plots |
| `confusion_matrix.png` | Per-class evaluation |
| `prediction_result.png` | Single image test output |
